### Libraries



In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.3 MB/s eta 0:00:00


### Login to huggingface

In [2]:
login()

### Testing

In [3]:
# === Use GPU if available ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# === Load tokenizer and model with LoRA ===
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, lora_repo_id)
model = model.to(device)
model.eval()

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [11]:
csv_path = hf_hub_download(
    repo_id="eduhuemar001/sentiment-GermEval2017",
    filename="germeval2017_cleaned.csv",
    repo_type="dataset"
)

# === Load 200 labeled GermEval examples ===
df = pd.read_csv(csv_path)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]

# Take 200 random examples
df = df.sample(n=200, random_state=42).reset_index(drop=True)

# === Prompt template ===
instruction = (
    "### Instruction:\n"
    "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
    "### Bewertung:\n"
)
answer_prefix = "\n\n### Antwort:\n"

# === Run model inference on 200 examples ===
model = model.to(device)
model.eval()

true_labels = []
pred_labels = []

for i, row in df.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract prediction
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    # Record results
    true_labels.append(true_label)
    pred_labels.append(answer if answer in ["positive", "neutral", "negative"] else "neutral")

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")

# === Classification report ===
print("\n=== Klassifikationsbericht ===")
print(classification_report(true_labels, pred_labels, digits=3))


[1] Bewertung: @DB_Bahn Kurz mal Taxi-Rechner.de bemüht. Mit Trinkgeld liegen wir bei 80€ von Rheine in die Emsmetropole
   Wahre Stimmung: neutral
   Modellantwort: neutral

[2] Bewertung: RT @therealmoneyboy: ich rede nicht von der Deutschen Bahn wenn ich sage das ich 1 Zug nehme die rede ist von 1 marihuana zigarette
   Wahre Stimmung: neutral
   Modellantwort: neutral

[3] Bewertung: zak.de Schienenersatzverkehr: Bus statt Bahn auf Hechinger HzL-Strecke Wichtiger Hinweis für die Pendler, die zwischen Hechingen und Gammertingen mit dem Zug fahren. (bs) 19.05.2016 - Fahrgäste müssen sich vom 23. bis 25. Mai auf Schienenersatzverkehr zwischen Hechingen und Gammertingen einstellen. Grund hierfür sind Gleisarbeiten.
   Wahre Stimmung: neutral
   Modellantwort: negative

[4] Bewertung: AW: ?? BYM-SILVESTER ?????? meingott die nachbarn haben seeed aufgedreht, es wird echt zeit, die nächste bahn zu nehmen
   Wahre Stimmung: neutral
   Modellantwort: neutral

[5] Bewertung: BOGESTRA - Boch

KeyboardInterrupt: 